In [ ]:
import random

#### **Motivation**

Solving Set Covering Problem by Max Min Ant System

At first, we should load data from file (e.g. `scpa2.txt`) and do the preprocessing.

In [ ]:
matrix = list()
costs = list()
row_num, column_num = 0, 0
with open('.\\scpa2.txt', 'r') as f:
    lines = f.readlines()

    step = 1
    count = 0
    rowcounter = 0
    for line in lines:
        l = list(map(int, line.split()))
        
        if step == 1:
            row_num = int(l[0])
            column_num = int(l[1])
            for _ in range(row_num):
                row = [0] * column_num
                matrix.append(row)
            step+=1
    
        elif step == 2:
            if len(costs) < column_num:
                costs += l
            if len(costs) == column_num:
                step+=1
        
        elif step == 3:
            if count == 0:
                count = l[0]
            else:
                for i in l:
                    matrix[rowcounter][i-1] = 1
                    count -=1
                    
                if count == 0:
                    rowcounter+=1
        

In [ ]:
class MaxMinAntSystem:
    def __init__(self, ants_num, iteration, tmin, tmax, t0, alpha, beta, q, rho =0.9):
        self.ants_num = ants_num
        self.iteration = iteration
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q = q
        self.min = tmin
        self.max = tmax
        self.phero = [t0 for _ in range(column_num)]
    
    def pheromone(self):
        print(self.phero)
        
    def MMAS(self):
        best = tuple()
        for it in range(self.iteration):
            
            # set cover list for each ant
            cover = []
            for j in range(self.ants_num):
                cover.append([0] * row_num)
            
                            
            for i in range(self.ants_num):
                positions = set()
                fitness = 0
                while True:
                    sum_phero = 0
                    prob_weight = []
                    for j in range(column_num):
                        if j in positions:
                            prob_weight.append(0)
                            continue
                        count = 0
                        for k in range(row_num):
                            if matrix[k][j] == 1 and cover[i][k] == 0:
                                count +=1
                        sum_phero += (self.phero[j]**self.alpha) * ((count/costs[j])**self.beta)
                        prob_weight.append((self.phero[j]**self.alpha) * ((count/costs[j])**self.beta))
                    
                
                    for j in range(column_num):
                        prob_weight[j] /= sum_phero
                    
                    position = random.choices(range(column_num), k = 1, weights = prob_weight)
                    p = position[0]
                    if p in positions:
                        continue
                    
                    positions.add(p)
                    fitness += costs[p]
                    for j in range(row_num):
                        if matrix[j][p] == 1:
                            cover[i][j] = 1
                            
                    # checking stopping condition
                    for j in range(row_num):
                        if cover[i][j] == 0:
                            break
                    else:
                        if i==0 and it==0:
                            best = (fitness, positions)
                        else:
                            if fitness < best[0]:
                                best = (fitness, positions)
                        break
                
            # decrease pheromone from all sets
            for i in range(column_num):
                self.phero[i] *= self.rho
                            
            # put pheromone for best solution
            for i in best[1]:
                self.phero[i] += self.q/best[0]
            
            # updating pheromones by max-min-ant-system formula 
            for i in range(column_num):
                self.phero[i] = max(self.min, min(self.phero[i], self.max))

            print(best)

#### **Setting Hyperparameters**

**$\alpha$** = 2 <br>
**$\beta$** = 6 <br>
**$\rho$** = 0.9 <br>
**$\tau_{0}$** = 3 <br>
**$\tau_{min}$** = 1 <br>
**$\tau_{max}$** = 40 <br>
**$Q$** = 100

- By running the algorithm for different values, I came to this values.e.g.

In [ ]:
x = MaxMinAntSystem(ants_num=5, iteration=30, alpha=2, beta=6, rho=.9, tmin=1, tmax=40, t0=3, q=100)
x.MMAS()

<table>
    <tr>
        <td>Testcase</td>
        <td>Mean of 10 run</td>
        <td>Goal</td>
    </tr>
    <tr>
        <td>scp41.txt</td>
        <td>445.6</td>
        <td>440</td>
    </tr>
    <tr>
        <td>scp51.txt</td>
        <td>269.2</td>
        <td>265</td>
    </tr>
    <tr>
        <td>scp54.txt</td>
        <td>250</td>
        <td>250</td>
    </tr>
    <tr>
        <td>scpa2.txt</td>
        <td>264.2</td>
        <td>260</td>
    </tr>
    <tr>
        <td>scpb1.txt</td>
        <td>71</td>
        <td>69</td>
    </tr>
</table>